In [ ]:
base    = 'CR'
variant = 'VAR-MS'
cluster = '33'
resolution = '2H'
horizon = [2025, 2030, 2035, 2040, 2045, 2050]
countries_of_interest = ['DE','GB','IT','FR','PL','ES','NL','CZ','BE','DK']
save_fig = True

---
---
### $\text{Load the results}$ 
---
---

In [ ]:
import pandas as pd
import pypsa
import numpy
import pickle
import numpy as np
pd.set_option('display.max_rows', None)
import yaml

In [ ]:
network_bl = {}
if base == 'baseline': 
    path_bl = f'results/{base}/'
else: 
    path_bl = f'results/RFNBO_{base}/'
for i in horizon:
    network_bl[i] = pypsa.Network(f'{path_bl}networks/base_s_{cluster}__{resolution}_{i}.nc')
    network_bl[i].buses.loc[network_bl[i].buses.index.str.contains('EU'),'country'] = 'EU'
if base == 'baseline': 
    with open(f"config/config.{base}.yaml", "r") as f:
        config_bl = yaml.safe_load(f)
else:
    with open(f"config/config.RFNBO_{base}.yaml", "r") as f:
        config_bl = yaml.safe_load(f)

In [ ]:
network_vr = {}
path_vr = f'results/RFNBO_{variant}/'
if base == 'baseline':
    path_dt = f'results/{base}_vs_RFNBO_{variant}/'
else:
    path_dt = f'results/RFNBO_{base}_vs_RFNBO_{variant}/'
for i in horizon:
    network_vr[i] = pypsa.Network(f'{path_vr}networks/base_s_{cluster}__{resolution}_{i}.nc')
    network_vr[i].buses.loc[network_vr[i].buses.index.str.contains('EU'),'country'] = 'EU'
with open(f"config/config.RFNBO_{variant}.yaml", "r") as f:
    config_vr = yaml.safe_load(f)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
colors = ['#2196F3', '#E91E63', '#4CAF50', '#9C27B0', '#795548', '#FFC107', '#F44336', '#009688', '#607D8B', '#00BCD4',  '#FF5722', '#FF9800']
import os
if not os.path.exists(f'{path_bl}plots/') and save_fig:
    os.mkdir(f'{path_bl}plots/')

In [ ]:
if not os.path.exists(f'{path_vr}plots/') and save_fig:
    os.mkdir(f'{path_vr}plots/')
if not os.path.exists(f'{path_dt}/') and save_fig:
    os.mkdir(f'{path_dt}/')

---
---
### $\text{Routines and helpers}$ 
---
---

In [ ]:
def get_total_investment_cost(n):
    """
    Compute total investment (capital) cost for a single PyPSA-Eur network.

    Mirrors the logic of `Investment_costs` (capital cost summed over
    generators, links, stores, storage units, lines) but works directly
    off the network object `n` instead of a pre-computed nodal_costs.csv,
    and returns both a per-country breakdown and an EU-wide total.

    Parameters
    ----------
    n : pypsa.Network

    Returns
    -------
    investment_cost_per_country : dict
        {country_code: total_capital_cost}
    investment_cost_total : float
        Sum of capital costs across all components in the network.
    """
    components_with_investment = {
        "Generator": n.generators,
        "Link": n.links,
        "Store": n.stores,
        "StorageUnit": n.storage_units,
        "Line": n.lines,
    }

    # column names differ slightly by component (p_nom_opt vs e_nom_opt)
    opt_capacity_col = {
        "Generator": "p_nom_opt",
        "Link": "p_nom_opt",
        "Store": "e_nom_opt",
        "StorageUnit": "p_nom_opt",
        "Line": "s_nom_opt",
    }

    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    countries_list = countries_list[countries_list != 'EU']

    investment_cost_per_country = {country: 0.0 for country in countries_list}
    investment_cost_total = 0.0

    for comp_name, df in components_with_investment.items():
        if df.empty or "capital_cost" not in df.columns:
            continue

        cap_col = opt_capacity_col[comp_name]
        if cap_col not in df.columns:
            continue

        capital_cost_series = df[cap_col] * df["capital_cost"]

        # attribute each asset's cost to a country based on its bus prefix
        if "bus" in df.columns:
            bus_ref = df["bus"]
        elif "bus0" in df.columns:
            bus_ref = df["bus0"]
        else:
            bus_ref = None

        for idx, cost in capital_cost_series.items():
            if pd.isna(cost):
                continue
            investment_cost_total += cost

            if bus_ref is not None:
                bus_name = bus_ref.loc[idx]
                country_code = bus_name[:2]
                if country_code in investment_cost_per_country:
                    investment_cost_per_country[country_code] += cost

    return investment_cost_per_country, investment_cost_total

In [ ]:
def get_total_investment_cost_vre(n, tech_list_vre=None):
    """
    Compute total investment (capital) cost for a single PyPSA-Eur network,
    restricted to a given list of technologies (by `carrier`).

    Mirrors the logic of `Investment_costs` (capital cost summed over
    generators, links, stores, storage units, lines) but works directly
    off the network object `n` instead of a pre-computed nodal_costs.csv,
    and returns both a per-country breakdown and an EU-wide total.

    Parameters
    ----------
    n : pypsa.Network
    tech_list_vre : list of str, optional
        List of carrier/technology names to include. If None, defaults to
        the VRE tech list.

    Returns
    -------
    investment_cost_per_country : dict
        {country_code: total_capital_cost}
    investment_cost_total : float
        Sum of capital costs across all components in the network,
        restricted to `tech_list_vre`.
    """
    if tech_list_vre is None:
        tech_list_vre = [
            'onwind', 'solar-hsat', 'offwind-float', 'offwind-dc',
            'solar', 'offwind-ac', 'ror', 'solar rooftop',
        ]

    components_with_investment = {
        "Generator": n.generators,
        "Link": n.links,
        "Store": n.stores,
        "StorageUnit": n.storage_units,
        "Line": n.lines,
    }

    # column names differ slightly by component (p_nom_opt vs e_nom_opt)
    opt_capacity_col = {
        "Generator": "p_nom_opt",
        "Link": "p_nom_opt",
        "Store": "e_nom_opt",
        "StorageUnit": "p_nom_opt",
        "Line": "s_nom_opt",
    }

    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    countries_list = countries_list[countries_list != 'EU']

    investment_cost_per_country = {country: 0.0 for country in countries_list}
    investment_cost_total = 0.0

    for comp_name, df in components_with_investment.items():
        if df.empty or "capital_cost" not in df.columns:
            continue

        cap_col = opt_capacity_col[comp_name]
        if cap_col not in df.columns:
            continue

        # restrict to the technologies of interest
        if "carrier" not in df.columns:
            continue
        df = df[df["carrier"].isin(tech_list_vre)]
        if df.empty:
            continue

        capital_cost_series = df[cap_col] * df["capital_cost"]

        # attribute each asset's cost to a country based on its bus prefix
        if "bus" in df.columns:
            bus_ref = df["bus"]
        elif "bus0" in df.columns:
            bus_ref = df["bus0"]
        else:
            bus_ref = None

        for idx, cost in capital_cost_series.items():
            if pd.isna(cost):
                continue
            investment_cost_total += cost

            if bus_ref is not None:
                bus_name = bus_ref.loc[idx]
                country_code = bus_name[:2]
                if country_code in investment_cost_per_country:
                    investment_cost_per_country[country_code] += cost

    return investment_cost_per_country, investment_cost_total

In [ ]:
def get_co2(n):
    
    co2_links_bus1 = n.links[
        n.links.bus1.str.contains('co2 atmosphere', case=False, na=False) 
    ]
    co2_links_bus2 = n.links[
        n.links.bus2.str.contains('co2 atmosphere', case=False, na=False) 
    ]
    co2_links_bus3 = n.links[
        n.links.bus3.str.contains('co2 atmosphere', case=False, na=False) 
    ]

    # CO2 from bus1 links
    if not co2_links_bus1.empty:
        co2_bus1 = (
            -n.links_t.p1[co2_links_bus1.index]
            .multiply(n.snapshot_weightings.generators, axis=0)
        )
    # CO2 from bus2 links
    if not co2_links_bus2.empty:
        co2_bus2 = (
            -n.links_t.p2[co2_links_bus2.index]
            .multiply(n.snapshot_weightings.generators, axis=0)
        )
    # CO2 from bus3 links
    if not co2_links_bus3.empty:
        co2_bus3 = (
            -n.links_t.p3[co2_links_bus3.index]
            .multiply(n.snapshot_weightings.generators, axis=0)
        )

    co2_all_buses = pd.concat((co2_bus1, co2_bus2, co2_bus3), axis=1).sum(axis = 0)

    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    co2_dict = {country: 0 for country in countries_list}

    for i in co2_all_buses.index:
        co2_dict[i[:2]] += co2_all_buses.loc[i]

    return co2_dict

In [ ]:
def get_electricity(n):
    electricity_buses_names = n.buses[(n.buses.carrier == 'AC')].index

    electricity_prices_per_bus = n.buses_t['marginal_price'][electricity_buses_names].mean( axis = 0)
    
    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    countries_list = countries_list[countries_list != 'EU']

    electricity_prices_per_country_dict = {country: [] for country in countries_list}

    electricity_prices_per_bus_dict = dict(electricity_prices_per_bus)

    for i in electricity_prices_per_bus.index:
        electricity_prices_per_country_dict[i[:2]].append(electricity_prices_per_bus.loc[i])
    for key, item in electricity_prices_per_country_dict.items():
        electricity_prices_per_country_dict[key] = np.mean(item)

    return electricity_prices_per_country_dict, electricity_prices_per_bus_dict

In [ ]:
def _get_duals_from_gc(co2_rows, countries_list):
    co2_prices_per_country = {ct: [] for ct in countries_list}

    for name, row in co2_rows.iterrows():
        ct = name[-2:]
        if ct in co2_prices_per_country:
            co2_prices_per_country[ct].append(row["mu"])

    co2_prices_per_country = {
        ct: np.mean(vals) * -1 if vals else np.nan
        for ct, vals in co2_prices_per_country.items()
    }
    return co2_prices_per_country


def _get_duals_from_model(n, countries_list):
    co2_prices_per_country = {ct: [] for ct in countries_list}

    for ct in countries_list:
        ct_duals = []
        for snapshot in n.snapshots:
            constraint_name = f"GlobalConstraint-co2_limit_per_country{ct}"
            try:
                dual_val = n.model.dual[constraint_name].values
                ct_duals.append(float(dual_val))
            except KeyError:
                pass

        co2_prices_per_country[ct] = np.mean(ct_duals) * -1 if ct_duals else np.nan

    return co2_prices_per_country

def get_co2_price(n):
    countries_list = n.buses.country.unique()
    countries_list = countries_list[
        (countries_list != '') & (countries_list != 'EU')
    ]

    gc = n.global_constraints
    co2_rows = gc[gc.index.str.contains("co2_limit_per_country", case=False)]

    # No constraint found → no CO2 price (e.g. 2025 baseline year)
    if co2_rows.empty:
        return {ct: 0.0 for ct in countries_list}

    if co2_rows["mu"].isna().all() or (co2_rows["mu"] == 0).all():
        print("mu column empty — reading duals directly from n.model.dual")
        return _get_duals_from_model(n, countries_list)
    else:
        return _get_duals_from_gc(co2_rows, countries_list)

In [ ]:
def get_hydrogen(n):
    hydrogen_buses_names = n.buses[(n.buses.carrier == 'H2')].index

    hydrogen_prices_per_bus = n.buses_t['marginal_price'][hydrogen_buses_names].mean( axis = 0)
    
    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    countries_list = countries_list[countries_list != 'EU']

    hydrogen_prices_per_country_dict = {country: [] for country in countries_list}

    hydrogen_prices_per_bus_dict = dict(hydrogen_prices_per_bus)

    for i in hydrogen_prices_per_bus.index:
        hydrogen_prices_per_country_dict[i[:2]].append(hydrogen_prices_per_bus.loc[i])
    for key, item in hydrogen_prices_per_country_dict.items():
        hydrogen_prices_per_country_dict[key] = np.mean(item)

    return hydrogen_prices_per_country_dict, hydrogen_prices_per_bus_dict

In [ ]:
def get_hydrogen_grid_connected(n):

    weights = n.snapshot_weightings.generators

    # 1. Isolate the electrolyzer links
    electrolyzers = n.links[n.links.carrier == 'H2 Electrolysis']
    elec_indices = electrolyzers.index

    # 2. Calculate Annualized Fixed Costs (CAPEX + Fixed O&M) per link
    # capital_cost in PyPSA is already annualized per MW
    fixed_costs = electrolyzers['capital_cost'] * electrolyzers['p_nom_opt']

    # 3. Calculate Variable input costs (Electricity consumed * Local Nodal Price)
    # Extract hourly electricity costs for only those specific buses
    link_to_bus_map = n.links.loc[elec_indices, 'bus0']
    aligned_prices = n.buses_t.marginal_price[link_to_bus_map]
    aligned_prices.columns = elec_indices
    # Extract hourly power consumption at bus0 for each electrolyzer
    hourly_consumption = n.links_t.p0[elec_indices]


    # Multiply element-wise (hourly) and sum over the year to get total variable cost per link
    variable_costs = (hourly_consumption.mul(weights, axis=0) * aligned_prices).sum(axis=0)

    # 4. Calculate total annual hydrogen production per link (Output at bus1)
    # Note: If your model tracks efficiency losses, p1 is the actual H2 generated
    annual_h2_produced = -n.links_t.p1[elec_indices].mul(weights, axis=0).sum(axis=0)

    # 5. Calculate LCOH per individual link
    # Avoid division by zero for links that weren't built (p_nom_opt == 0)
    lcoh_per_link = (fixed_costs + variable_costs) / annual_h2_produced
    lcoh_per_link = lcoh_per_link.dropna()  # Drops links that were not optimized into existence

    lcoh_per_link_capex = fixed_costs / annual_h2_produced
    lcoh_per_link_capex = lcoh_per_link_capex.dropna()  # Drops links that were not optimized into existence
    lcoh_per_link_opex = variable_costs / annual_h2_produced
    lcoh_per_link_opex = lcoh_per_link_opex.dropna()  # Drops links that were not optimized into existence

    # Convert to a clean DataFrame for analysis or plotting
    df_h2 = pd.DataFrame({
        'Bus_Location': electrolyzers.loc[lcoh_per_link.index, 'bus1'],
        'Capacity': electrolyzers.loc[lcoh_per_link.index, 'p_nom_opt'],
        'H2 Produced' : annual_h2_produced,
        'LCOH': lcoh_per_link,
        'LCOH_capex': lcoh_per_link_capex,
        'LCOH_opex': lcoh_per_link_opex
    })

    return df_h2

In [ ]:
def get_hydrogen_smr(n):

    weights = n.snapshot_weightings.generators

    # 1. Isolate the SMR links
    # Note: adjust carrier name if your model distinguishes 'SMR' vs 'SMR CC' (with carbon capture)
    smr_links = n.links[n.links.carrier == 'SMR']
    smr_indices = smr_links.index

    # 2. Calculate Annualized Fixed Costs (CAPEX + Fixed O&M) per link
    fixed_costs = smr_links['capital_cost'] * smr_links['p_nom_opt']

    # 3. Calculate Variable input costs
    # 3a. Fuel cost: gas consumed (bus0) * local gas nodal price
    link_to_bus_map = smr_links['bus0']
    aligned_gas_prices = n.buses_t.marginal_price[link_to_bus_map]
    aligned_gas_prices.columns = smr_indices

    hourly_gas_consumption = n.links_t.p0[smr_indices]

    fuel_costs = (hourly_gas_consumption.mul(weights, axis=0) * aligned_gas_prices).sum(axis=0)

    # 3b. Marginal cost component (PyPSA convention: marginal_cost applies per unit of p0)
    # SMR links often carry a non-zero marginal_cost even though fuel is priced via bus0
    marginal_costs = (hourly_gas_consumption.mul(weights, axis=0) * smr_links['marginal_cost']).sum(axis=0)

    variable_costs = fuel_costs + marginal_costs

    # 4. Calculate total annual hydrogen production per link (Output at bus1)
    annual_h2_produced = -n.links_t.p1[smr_indices].mul(weights, axis=0).sum(axis=0)

    # 5. Calculate LCOH per individual link
    lcoh_per_link = (fixed_costs + variable_costs) / annual_h2_produced
    lcoh_per_link = lcoh_per_link.dropna()  # Drops links that were not optimized into existence

    # Convert to a clean DataFrame for analysis or plotting
    df_h2 = pd.DataFrame({
        'Bus_Location': smr_links.loc[lcoh_per_link.index, 'bus1'],
        'Capacity': smr_links.loc[lcoh_per_link.index, 'p_nom_opt'],
        'H2 Produced': annual_h2_produced,
        'LCOH': lcoh_per_link
    })

    return df_h2

In [ ]:

def get_vre_share_carbon_intensity(n, config, config_name, year, country):
    '''
    This function gets the VRE share and co2 intensity (g/Kwh) in electricity grid power supply
    from the previous optimised planning horizon which is further used in the additionality constraint. 
    '''
    
    #/home/alaterre/pypsa-eur_RFNBO/resources/baseline/costs_2025_processed.csv

    params_file = pd.read_csv(f"/home/alaterre/pypsa-eur_RFNBO/resources/{config_name}/costs_{year}_processed.csv", index_col=[0, 1]).sort_index()
    co2_intensity_raw = params_file["CO2 intensity"]
    co2_intensity_raw.index = co2_intensity_raw.index.droplevel(1)
    #convert t/MWh to g/kWh
    co2_intensity_g_kwh = co2_intensity_raw * 1000
    
    generator_types = list(
      set(config["electricity"]["renewable_carriers"] + ["solar rooftop","ror"]))

    conv_types = list(
      set(config["electricity"]["conventional_carriers"] + ["urban central gas CHP",
            "urban central gas CHP CC","urban central solid biomass CHP","urban central solid biomass CHP CC",
            "H2 Fuel Cell","H2 turbine","geothermal organic rankine cycle"]))
    gens = n.generators.index[
      n.generators.carrier.isin(generator_types)
    ]

    links = n.links.index[
      n.links.carrier.isin(conv_types)
    ]
    
    hydro= n.storage_units.index[
      n.storage_units.carrier == "hydro"
    ]
    
    gen = (
      (n.snapshot_weightings.generators @ n.generators_t.p[gens])
      .filter(like=country)
      .groupby(
        [
            n.generators.loc[gens, "carrier"],
        ]
     )
    .sum()
    .mul(1e3)   #convert MWh to kWh
     )
    link = (
    (n.snapshot_weightings.generators @ -n.links_t.p1[links])
    .filter(like=country)
    .groupby(
        [
            n.links.loc[links, "carrier"],
        ]
    )
    .sum()
    .mul(1e3)   #convert MWh to kWh
    )
    hyd = (
      (n.snapshot_weightings.generators @ n.storage_units_t.p_dispatch[hydro])
      .filter(like=country)
      .groupby(
        [
            n.storage_units.loc[hydro, "carrier"],
        ]
     )
    .sum()
    .mul(1e3)   #convert MWh to kWh
     )
    tota_elec_grid_techs = pd.concat([gen, link, hyd])
    #align with grid carriers
    co2_intensity_g_kwh = co2_intensity_g_kwh.reindex(tota_elec_grid_techs.index).fillna(0)
    
    gas_chp_types = [
        "urban central gas CHP",
        "urban central gas CHP CC",
    ]

    existing_types = [t for t in gas_chp_types if t in co2_intensity_g_kwh.index]
    co2_intensity_g_kwh.loc[existing_types] = co2_intensity_g_kwh.loc["CCGT"]

    #mapping emissions
    gen_consumption = (
        (n.snapshot_weightings.generators @ n.generators_t.p[gens])
        .div(n.generators.loc[gens, "efficiency"])
        .filter(like=country).mul(1e3)
    )
    
    gen_emissions = (
        gen_consumption
        * n.generators.loc[gen_consumption.index, "carrier"].map(co2_intensity_g_kwh)
    )
    
    gen_emissions = (
        gen_emissions
        .groupby(n.generators.loc[gen_emissions.index, "carrier"])
        .sum()
    )
    
    link_consumption = (
        (n.snapshot_weightings.generators @ -n.links_t.p1[links])
        .div(n.links.loc[links, "efficiency"])
        .filter(like=country).mul(1e3)
    )

    link_emissions = (link_consumption
                     * n.links.loc[link_consumption.index, "carrier"].map(co2_intensity_g_kwh))
    
    
    link_production = (
        (n.snapshot_weightings.generators @ -n.links_t.p1[links])
        .filter(like=country).mul(1e3)
        .groupby(n.links.loc[link_emissions.index, "carrier"])
        .sum()
    ) 

    link_emissions = (
        link_emissions
        .groupby(n.links.loc[link_emissions.index, "carrier"])
        .sum())
    
    link_footprint = (link_emissions
                      .div(link_production)
    )
    #print(link_footprint)

    emissions = pd.concat(
        [gen_emissions, link_emissions,]
    ).groupby(level=0).sum()
    
    #convert to g/MJ
    emissions = emissions / 3.6
    total_emissions = emissions.sum()

    renewable_carriers = generator_types + [
    "urban central solid biomass CHP",
    "urban central solid biomass CHP CC",
    "geothermal organic rankine cycle"]

    renewable_total = tota_elec_grid_techs[
    tota_elec_grid_techs.index.isin(renewable_carriers)
    ].sum()
    
    total_generation = tota_elec_grid_techs.sum()
    
    renewable_share = 100 * renewable_total / total_generation
    total_co2_intensity = total_emissions / total_generation
 
    return {
        "country": country,
        "renewable_share": renewable_share,
        "co2_intensity": total_co2_intensity,
    }   

In [ ]:
def get_vre_share_carbon_intensity_prod(n, config, config_name, year):
    '''
    This function gets the VRE share and co2 intensity (g/kWh) in electricity grid power supply
    from the previous optimised planning horizon which is further used in the additionality constraint.
    Returns a dictionary keyed by country with renewable_share and co2_intensity.
    '''

    params_file = pd.read_csv(
        f"/home/alaterre/pypsa-eur_RFNBO/resources/{config_name}/costs_{year}_processed.csv",
        index_col=[0, 1]
    ).sort_index()
    co2_intensity_raw = params_file["CO2 intensity"]
    co2_intensity_raw.index = co2_intensity_raw.index.droplevel(1)
    co2_intensity_g_kwh = co2_intensity_raw * 1000  # convert t/MWh to g/kWh

    generator_types = list(
        set(config["electricity"]["renewable_carriers"] + ["solar rooftop", "ror"])
    )
    conv_types = list(
        set(config["electricity"]["conventional_carriers"] + [
            "urban central gas CHP", "urban central gas CHP CC",
            "urban central solid biomass CHP", "urban central solid biomass CHP CC",
            "H2 Fuel Cell", "H2 turbine", "geothermal organic rankine cycle"
        ])
    )
    renewable_carriers = generator_types + [
        "urban central solid biomass CHP",
        "urban central solid biomass CHP CC",
        "geothermal organic rankine cycle"
    ]
    gas_chp_types = ["urban central gas CHP", "urban central gas CHP CC"]

    gens = n.generators.index[n.generators.carrier.isin(generator_types)]
    links = n.links.index[n.links.carrier.isin(conv_types)]
    hydro = n.storage_units.index[n.storage_units.carrier == "hydro"]

    countries = [c for c in n.buses.country.unique() if c not in ("EU", "")]

    # --- Weighted production time series (all buses) ---
    gen_p = n.snapshot_weightings.generators @ n.generators_t.p[gens]         # index: bus
    link_p = n.snapshot_weightings.generators @ -n.links_t.p1[links]          # index: bus
    hyd_p = n.snapshot_weightings.generators @ n.storage_units_t.p_dispatch[hydro]  # index: bus

    # --- Weighted fuel consumption time series (for emissions) ---
    gen_consumption = (
        (n.snapshot_weightings.generators @ n.generators_t.p[gens])
        .div(n.generators.loc[gens, "efficiency"])
        .mul(1e3)
    )
    link_consumption = (
        (n.snapshot_weightings.generators @ -n.links_t.p1[links])
        .div(n.links.loc[links, "efficiency"])
        .mul(1e3)
    )

    results = {}

    for country in countries:
        # --- Filter to country ---
        gen_c = (
            gen_p.filter(like=country)
            .groupby(n.generators.loc[gens, "carrier"])
            .sum()
            .mul(1e3)
        )
        link_c = (
            link_p.filter(like=country)
            .groupby(n.links.loc[links, "carrier"])
            .sum()
            .mul(1e3)
        )
        hyd_c = (
            hyd_p.filter(like=country)
            .groupby(n.storage_units.loc[hydro, "carrier"])
            .sum()
            .mul(1e3)
        )

        total_elec = pd.concat([gen_c, link_c, hyd_c])

        # --- CO2 intensity aligned to carriers present ---
        co2_c = co2_intensity_g_kwh.reindex(total_elec.index).fillna(0)
        existing_chp = [t for t in gas_chp_types if t in co2_c.index]
        if existing_chp and "CCGT" in co2_intensity_g_kwh.index:
            co2_c.loc[existing_chp] = co2_intensity_g_kwh.loc["CCGT"]

        # --- Emissions ---
        gen_em = (
            gen_consumption.filter(like=country)
            * n.generators.loc[gen_consumption.filter(like=country).index, "carrier"]
              .map(co2_c)
        )
        gen_em = gen_em.groupby(
            n.generators.loc[gen_em.index, "carrier"]
        ).sum()

        link_em = (
            link_consumption.filter(like=country)
            * n.links.loc[link_consumption.filter(like=country).index, "carrier"]
              .map(co2_c)
        )
        link_em = link_em.groupby(
            n.links.loc[link_em.index, "carrier"]
        ).sum()

        emissions = pd.concat([gen_em, link_em]).groupby(level=0).sum()
        emissions = emissions / 3.6  # convert g/kWh to g/MJ

        # --- Shares ---
        total_generation = total_elec.sum()
        renewable_total = total_elec[total_elec.index.isin(renewable_carriers)].sum()

        results[country] = {
            "renewable_share": 100 * renewable_total / total_generation if total_generation > 0 else 0,
            "co2_intensity": emissions.sum() / total_generation if total_generation > 0 else 0,
        }

    return results


In [ ]:
def get_vre_share_carbon_intensity_cons(n, config, config_name, year):
    '''
    Computes consumption-based renewable share and CO2 intensity per country,
    accounting for cross-border electricity trade (AC lines + DC links).
    
    Steps:
      1. Get production-based renewable share and CO2 intensity per country.
      2. Compute net imports from each trading partner.
      3. Weighted average: self-consumption * domestic intensity + imports * exporter intensity.
    '''

    # --- Step 1: Production-based metrics ---
    prod_based = get_vre_share_carbon_intensity_prod(n, config, config_name, year)
    countries = list(prod_based.keys())

    # --- Step 2: Cross-border net flows (AC lines + DC links) ---
    # net_imports[c_importer][c_exporter] = net MWh imported by c_importer from c_exporter
    net_imports = {c: {c2: 0 for c2 in countries} for c in countries}

    # AC lines
    for line in n.lines.index:
        b0, b1 = n.lines.at[line, "bus0"], n.lines.at[line, "bus1"]
        c0 = n.buses.at[b0, "country"]
        c1 = n.buses.at[b1, "country"]
        if c0 == c1 or c0 not in countries or c1 not in countries:
            continue
        flow = (n.snapshot_weightings.generators @ n.lines_t.p0[line])  # positive = c0 -> c1
        net_imports[c1][c0] += flow
        net_imports[c0][c1] -= flow

    # DC links
    dc_links = n.links.index[n.links.carrier == "DC"]
    for link in dc_links:
        b0, b1 = n.links.at[link, "bus0"], n.links.at[link, "bus1"]
        c0 = n.buses.at[b0, "country"]
        c1 = n.buses.at[b1, "country"]
        if c0 == c1 or c0 not in countries or c1 not in countries:
            continue
        flow = (n.snapshot_weightings.generators @ -n.links_t.p1[link])  # positive = net into c1
        net_imports[c1][c0] += flow
        net_imports[c0][c1] -= flow

    # --- Step 3: Consumption-based weighted average ---
    results = {}

    for c in countries:
        domestic_ren_share = prod_based[c]["renewable_share"] / 100
        domestic_co2 = prod_based[c]["co2_intensity"]

        # Total imports from each partner (only positive = actual net imports)
        total_imports = sum(max(v, 0) for v in net_imports[c].values())

        # Gross production (used to estimate self-consumption)
        # self_consumption = gross_prod - exports = gross_prod - sum of positive outflows
        total_exports = sum(max(-v, 0) for v in net_imports[c].values())

        # Get domestic gross production from production-based function
        # Approximate: use total_elec computed inside prod_based indirectly via
        # renewable_share = ren / total => total = ren / share (if share > 0)
        # Instead, recompute gross production directly here
        generator_types = list(
            set(config["electricity"]["renewable_carriers"] + ["solar rooftop", "ror"])
        )
        conv_types = list(
            set(config["electricity"]["conventional_carriers"] + [
                "urban central gas CHP", "urban central gas CHP CC",
                "urban central solid biomass CHP", "urban central solid biomass CHP CC",
                "H2 Fuel Cell", "H2 turbine", "geothermal organic rankine cycle"
            ])
        )
        gens = n.generators.index[n.generators.carrier.isin(generator_types)]
        links = n.links.index[n.links.carrier.isin(conv_types)]
        hydro = n.storage_units.index[n.storage_units.carrier == "hydro"]

        gross_prod = (
            (n.snapshot_weightings.generators @ n.generators_t.p[gens]).filter(like=c).sum()
            + (n.snapshot_weightings.generators @ -n.links_t.p1[links]).filter(like=c).sum()
            + (n.snapshot_weightings.generators @ n.storage_units_t.p_dispatch[hydro]).filter(like=c).sum()
        )

        self_consumption = gross_prod - total_exports
        total_consumption = self_consumption + total_imports

        if total_consumption <= 0:
            results[c] = {"renewable_share": 0, "co2_intensity": 0}
            continue

        # Weighted sum over self-consumption and each net import source
        weighted_ren = self_consumption * domestic_ren_share
        weighted_co2 = self_consumption * domestic_co2

        for c2, net_imp in net_imports[c].items():
            if net_imp > 0:  # actual net import from c2
                weighted_ren += net_imp * (prod_based[c2]["renewable_share"] / 100)
                weighted_co2 += net_imp * prod_based[c2]["co2_intensity"]

        results[c] = {
            "renewable_share": 100 * weighted_ren / total_consumption,
            "co2_intensity": weighted_co2 / total_consumption,
        }

    return results

---
---
### $\text{Cost of regulation}$ 
---
---

In [ ]:
invest_dict_all_years_bl = {year : {} for year in horizon}
total_bl = {}
for year in horizon:
    invest_dict_all_years_bl[year], total_bl[year] = get_total_investment_cost(network_bl[year])

In [ ]:
invest_dict_all_years_vr = {year : {} for year in horizon}
invest_dict_all_years_dt = {year : {} for year in horizon}
total_vr = {}
total_dt = {}
for year in horizon:
    invest_dict_all_years_vr[year], total_vr[year] = get_total_investment_cost(network_vr[year])
    invest_dict_all_years_dt[year], total_dt[year] = (
        pd.Series(invest_dict_all_years_vr[year]) - pd.Series(invest_dict_all_years_bl[year])
    ).to_dict(), total_vr[year]-total_bl[year]

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    invest_country_dt = np.array([invest_dict[country] for invest_dict in invest_dict_all_years_dt.values()])
    invest_country_bl = np.array([invest_dict[country] for invest_dict in invest_dict_all_years_bl.values()])
    
    ax1.plot(horizon, [v for v in invest_country_dt / invest_country_bl * 100],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Relative Difference in Investment Costs [%]', fontsize=12)
ax1.set_title(f'Difference by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
invest_total_dt = np.array([invest_dict for invest_dict in total_dt.values()])
invest_total_bl = np.array([invest_dict for invest_dict in total_bl.values()])
ax2.plot(horizon, [v for v in invest_total_dt / invest_total_bl * 100],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Relative Difference in Investment Costs [%]', fontsize=12)
ax2.set_title(f'EU Total Difference (Cumulated = {sum(invest_total_dt) / sum(invest_total_bl) * 1e2:.1f}%)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Relative Difference in Total Investment Costs Over Time (= {variant} - {base})'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
invest_vre_dict_all_years_bl = {year : {} for year in horizon}
total_bl = {}
for year in horizon:
    invest_vre_dict_all_years_bl[year], total_bl[year] = get_total_investment_cost_vre(network_bl[year])

In [ ]:
invest_vre_dict_all_years_vr = {year : {} for year in horizon}
invest_vre_dict_all_years_dt = {year : {} for year in horizon}
total_vr = {}
total_dt = {}
for year in horizon:
    invest_vre_dict_all_years_vr[year], total_vr[year] = get_total_investment_cost_vre(network_vr[year])
    invest_vre_dict_all_years_dt[year], total_dt[year] = (
        pd.Series(invest_vre_dict_all_years_vr[year]) - pd.Series(invest_vre_dict_all_years_bl[year])
    ).to_dict(), total_vr[year]-total_bl[year]

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    invest_vre_country_dt = np.array([invest_vre_dict[country] for invest_vre_dict in invest_vre_dict_all_years_dt.values()])
    invest_vre_country_bl = np.array([invest_vre_dict[country] for invest_vre_dict in invest_vre_dict_all_years_bl.values()])
    
    ax1.plot(horizon, [v for v in invest_vre_country_dt / invest_vre_country_bl * 100],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Relative Difference in VRE Investment Costs [%]', fontsize=12)
ax1.set_title(f'Difference by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
invest_vre_total_dt = np.array([invest_vre_dict for invest_vre_dict in total_dt.values()])
invest_vre_total_bl = np.array([invest_vre_dict for invest_vre_dict in total_bl.values()])
ax2.plot(horizon, [v for v in invest_vre_total_dt / invest_vre_total_bl * 100],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Relative Difference in VRE Investment Costs [%]', fontsize=12)
ax2.set_title(f'EU Total Difference (Cumulated = {sum(invest_vre_total_dt) / sum(invest_vre_total_bl) * 1e2:.1f}%)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Relative Difference in VRE Investment Costs Over Time (= {variant} - {base})'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

---
---
### $\text{CO}_2 \text{ emissions and price}$ 
---
---

In [ ]:
co2_dict_all_years_bl = {year : {} for year in horizon}
for year in horizon:
    co2_dict_all_years_bl[year] = get_co2(network_bl[year])

In [ ]:
co2_dict_all_years_vr = {year : {} for year in horizon}
co2_dict_all_years_dt = {year : {} for year in horizon}
for year in horizon:
    co2_dict_all_years_vr[year] = get_co2(network_vr[year])
    co2_dict_all_years_dt[year] = (pd.Series(co2_dict_all_years_vr[year]) - pd.Series(co2_dict_all_years_bl[year])
    ).to_dict()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    co2_country = [co2_dict[country] for co2_dict in co2_dict_all_years_bl.values()]
    ax1.plot(horizon, [v / 1e6 for v in co2_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('CO₂ Emissions [MtCO₂]', fontsize=12)
ax1.set_title('CO₂ Emissions by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
co2_EU = [sum(co2_dict.values())-co2_dict['EU'] for co2_dict in co2_dict_all_years_bl.values()]
ax2.plot(horizon, [v / 1e6 for v in co2_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('CO₂ Emissions [MtCO₂]', fontsize=12)
ax2.set_title('EU Total CO₂ Emissions', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Net CO₂ Emissions Over Time ({base} scenario)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_bl}plots/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: baseline, countries ---
for i, country in enumerate(countries_of_interest):
    co2_country = [co2_dict[country] for co2_dict in co2_dict_all_years_bl.values()]
    ax1.plot(horizon, [v / 1e6 for v in co2_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('CO₂ Emissions [MtCO₂]', fontsize=12)
ax1.set_title(f'{base}', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: variant, countries ---
for i, country in enumerate(countries_of_interest):
    co2_country = [co2_dict[country] for co2_dict in co2_dict_all_years_vr.values()]
    ax2.plot(horizon, [v / 1e6 for v in co2_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('CO₂ Emissions [MtCO₂]', fontsize=12)
ax2.set_title(variant, fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# --- Overall title and save figure ---
title = 'Net CO₂ Emissions Over Time by Country (comparison)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
# if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: baseline, EU ---
co2_EU = [sum(co2_dict.values())-co2_dict['EU'] for co2_dict in co2_dict_all_years_bl.values()]
ax1.plot(horizon, [v / 1e6 for v in co2_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('CO₂ Emissions [MtCO₂]', fontsize=12)
ax1.set_title(f'{base}', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: variant, EU ---
co2_EU = [sum(co2_dict.values())-co2_dict['EU'] for co2_dict in co2_dict_all_years_vr.values()]
ax2.plot(horizon, [v / 1e6 for v in co2_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('CO₂ Emissions [MtCO₂]', fontsize=12)
ax2.set_title(variant, fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'Net CO₂ Emissions Over Time in EU (comparison)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
# if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    co2_country = [co2_dict[country] for co2_dict in co2_dict_all_years_dt.values()]
    ax1.plot(horizon, [v / 1e6 for v in co2_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Difference in CO₂ Emissions [MtCO₂]', fontsize=12)
ax1.set_title('Difference by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
co2_EU_bl = [sum(co2_dict.values())-co2_dict['EU'] for co2_dict in co2_dict_all_years_bl.values()]
co2_EU_dt = [sum(co2_dict.values())-co2_dict['EU'] for co2_dict in co2_dict_all_years_dt.values()]
ax2.plot(horizon, [v / 1e6 for v in co2_EU_dt],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Difference in CO₂ Emissions [MtCO₂]', fontsize=12)
ax2.set_title(f'EU Total Difference (Cumulated = {sum(co2_EU_dt) / 1e6:.0f} MtCO₂, {sum(co2_EU_dt) / sum(co2_EU_bl) * 1e2:.1f}%)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Difference in CO₂ Emissions Over Time (= {variant} - {base})'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    co2_country_dt = np.array([co2_dict[country] for co2_dict in co2_dict_all_years_dt.values()])
    co2_country_bl = np.array([co2_dict[country] for co2_dict in co2_dict_all_years_bl.values()])
    ax1.plot(horizon, co2_country_dt/co2_country_bl*100,
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Difference in CO₂ Emissions [%]', fontsize=12)
ax1.set_title('Difference by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
co2_EU_dt = np.array([sum(co2_dict.values())-co2_dict['EU'] for co2_dict in co2_dict_all_years_dt.values()])
co2_EU_bl = np.array([sum(co2_dict.values())-co2_dict['EU'] for co2_dict in co2_dict_all_years_bl.values()])
ax2.plot(horizon, co2_EU_dt/co2_EU_bl*100,
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Difference in CO₂ Emissions [%]', fontsize=12)
ax2.set_title(f'EU Total Difference (Cumulated = {sum(co2_EU_dt)/sum(co2_EU_bl)*100:.1f}%)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Relative Difference in CO₂ Emissions Over Time (= {variant} - {base})'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
# if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
co2_price_per_bus_dict_all_years_bl     = {year : {} for year in horizon}
co2_price_per_country_dict_all_years_bl = {year : {} for year in horizon}
for year in horizon:
    co2_price_per_country_dict_all_years_bl[year] = get_co2_price(network_bl[year])

In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(7, 6), sharey=True)

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    electricity_country = [electricity_dict[country] for electricity_dict in co2_price_per_country_dict_all_years_bl.values()]
    ax1.plot(horizon, [v for v in electricity_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('CO2 Prices [€/ton]', fontsize=12)
ax1.set_title('CO2 Prices by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

title = f'CO2 Prices Over Time ({base} scenario)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
# if save_fig: plt.savefig(f'{path_bl}plots/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

---
---
### $\text{Electricity Prices}$ 
---
---

In [ ]:
electricity_per_bus_dict_all_years_bl     = {year : {} for year in horizon}
electricity_per_country_dict_all_years_bl = {year : {} for year in horizon}
for year in horizon:
    electricity_per_country_dict_all_years_bl[year], electricity_per_bus_dict_all_years_bl[year] = get_electricity(network_bl[year])

In [ ]:
electricity_per_bus_dict_all_years_vr     = {year : {} for year in horizon}
electricity_per_bus_dict_all_years_dt     = {year : {} for year in horizon}
electricity_per_country_dict_all_years_vr = {year : {} for year in horizon}
electricity_per_country_dict_all_years_dt = {year : {} for year in horizon}
for year in horizon:
    electricity_per_country_dict_all_years_vr[year], electricity_per_bus_dict_all_years_vr[year] = get_electricity(network_vr[year])
    electricity_per_country_dict_all_years_dt[year], electricity_per_bus_dict_all_years_dt[year] = (pd.Series(electricity_per_country_dict_all_years_vr[year]) - pd.Series(electricity_per_country_dict_all_years_bl[year])).to_dict(), (pd.Series(electricity_per_bus_dict_all_years_vr[year]) - pd.Series(electricity_per_bus_dict_all_years_bl[year])).to_dict()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    electricity_country = [electricity_dict[country] for electricity_dict in electricity_per_country_dict_all_years_bl.values()]
    ax1.plot(horizon, [v for v in electricity_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Marginal Electricity Costs [€/MWh]', fontsize=12)
ax1.set_title('Marginal Costs by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
electricity_EU = [np.mean(list(electricity_dict.values())) for electricity_dict in electricity_per_bus_dict_all_years_bl.values()]
ax2.plot(horizon, [v for v in electricity_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU mean', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Marginal Electricity Costs [€/MWh]', fontsize=12)
ax2.set_title('EU Marginal Costs', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Marginal Electricity Costs Over Time ({base} scenario)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_bl}plots/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    electricity_country = [electricity_dict[country] for electricity_dict in electricity_per_country_dict_all_years_bl.values()]
    ax1.plot(horizon, [v for v in electricity_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Marginal Electricity Costs [€/MWh]', fontsize=12)
ax1.set_title(base, fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
for i, country in enumerate(countries_of_interest):
    electricity_country = [electricity_dict[country] for electricity_dict in electricity_per_country_dict_all_years_vr.values()]
    ax2.plot(horizon, [v for v in electricity_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Marginal Electricity Costs [€/MWh]', fontsize=12)
ax2.set_title(variant, fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'Marginal Electricity Costs Over Time by Country (comparison)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
# if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: baseline, EU ---
electricity_EU = [np.mean(list(electricity_dict.values())) for electricity_dict in electricity_per_bus_dict_all_years_bl.values()]
ax1.plot(horizon, [v for v in electricity_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU mean', linestyle='--')

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Marginal Electricity Costs [€/MWh]', fontsize=12)
ax1.set_title(base, fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: variant, EU ---
electricity_EU = [np.mean(list(electricity_dict.values())) for electricity_dict in electricity_per_bus_dict_all_years_vr.values()]
ax2.plot(horizon, [v for v in electricity_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU mean', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Marginal Electricity Costs [€/MWh]', fontsize=12)
ax2.set_title(variant, fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'Marginal Electricity Costs Over Time in EU (comparison)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
# if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    electricity_country = [electricity_dict[country] for electricity_dict in electricity_per_country_dict_all_years_dt.values()]
    ax1.plot(horizon, [v for v in electricity_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Difference in Marginal Electricity Costs [€/MWh]', fontsize=12)
ax1.set_title('Difference by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
electricity_EU = [np.mean(list(electricity_dict.values())) for electricity_dict in electricity_per_bus_dict_all_years_dt.values()]
ax2.plot(horizon, [v for v in electricity_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU mean', linestyle='--')

ax2.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Difference in Marginal Electricity Costs [€/MWh]', fontsize=12)
ax2.set_title('EU Mean Difference', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Difference in Marginal Electricity Costs Over Time (= {variant} - {base})'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    electricity_country_dt = np.array([electricity_dict[country] for electricity_dict in electricity_per_country_dict_all_years_dt.values()])
    electricity_country_bl = np.array([electricity_dict[country] for electricity_dict in electricity_per_country_dict_all_years_bl.values()])
    ax1.plot(horizon, electricity_country_dt / electricity_country_bl * 100,
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)
    
ax1.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Difference in Marginal Costs [%]', fontsize=12)
ax1.set_title('Difference by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
electricity_EU_dt = np.array([np.mean(list(electricity_dict.values())) for electricity_dict in electricity_per_bus_dict_all_years_dt.values()])
electricity_EU_bl = np.array([np.mean(list(electricity_dict.values())) for electricity_dict in electricity_per_bus_dict_all_years_bl.values()])
ax2.plot(horizon, electricity_EU_dt / electricity_EU_bl * 100,
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU mean', linestyle='--')

ax2.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Difference in Marginal Costs [%]', fontsize=12)
ax2.set_title('EU Mean Difference', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Relative Difference in Marginal Electricity Costs Over Time (= {variant} - {base})'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

---
---
### $\text{Molecules Prices}$ 
---
---

In [ ]:
hydrogen_per_bus_dict_all_years_bl     = {year : {} for year in horizon}
hydrogen_per_country_dict_all_years_bl = {year : {} for year in horizon}

for year in horizon:
    hydrogen_per_country_dict_all_years_bl[year], hydrogen_per_bus_dict_all_years_bl[year] = get_hydrogen(network_bl[year])

In [ ]:
hydrogen_per_bus_dict_all_years_vr     = {year : {} for year in horizon}
hydrogen_per_bus_dict_all_years_dt     = {year : {} for year in horizon}
hydrogen_per_country_dict_all_years_vr = {year : {} for year in horizon}
hydrogen_per_country_dict_all_years_dt = {year : {} for year in horizon}

for year in horizon:
    hydrogen_per_country_dict_all_years_vr[year], hydrogen_per_bus_dict_all_years_vr[year] = get_hydrogen(network_vr[year])
    hydrogen_per_country_dict_all_years_dt[year], hydrogen_per_bus_dict_all_years_dt[year] = (pd.Series(hydrogen_per_country_dict_all_years_vr[year]) - pd.Series(hydrogen_per_country_dict_all_years_bl[year])).to_dict(), (pd.Series(hydrogen_per_bus_dict_all_years_vr[year]) - pd.Series(hydrogen_per_bus_dict_all_years_bl[year])).to_dict()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    hydrogen_country = [hydrogen_dict[country] for hydrogen_dict in hydrogen_per_country_dict_all_years_bl.values()]
    ax1.plot(horizon, [v for v in hydrogen_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Hydrogen Prices [€/MWh]', fontsize=12)
ax1.set_title('Hydrogen Prices by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
hydrogen_EU = [np.mean(list(hydrogen_dict.values())) for hydrogen_dict in hydrogen_per_bus_dict_all_years_bl.values()]
ax2.plot(horizon, [v for v in hydrogen_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU mean', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Hydrogen Prices [€/MWh]', fontsize=12)
ax2.set_title('EU Mean Hydrogen Prices', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Global Hydrogen Prices Over Time ({base} scenario)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_bl}plots/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    hydrogen_country = [hydrogen_dict[country] for hydrogen_dict in hydrogen_per_country_dict_all_years_bl.values()]
    ax1.plot(horizon, [v for v in hydrogen_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Hydrogen Prices [€/MWh]', fontsize=12)
ax1.set_title(base, fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
for i, country in enumerate(countries_of_interest):
    hydrogen_country = [hydrogen_dict[country] for hydrogen_dict in hydrogen_per_country_dict_all_years_vr.values()]
    ax2.plot(horizon, [v for v in hydrogen_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Hydrogen Prices [€/MWh]', fontsize=12)
ax2.set_title(variant, fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'Global Hydrogen Prices Over Time by Country (comparison)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
# if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: baseline, EU ---
hydrogen_EU = [np.mean(list(hydrogen_dict.values())) for hydrogen_dict in hydrogen_per_bus_dict_all_years_bl.values()]
ax1.plot(horizon, [v for v in hydrogen_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Hydrogen Prices [€/MWh]', fontsize=12)
ax1.set_title(base, fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: variant, EU ---
hydrogen_EU = [np.mean(list(hydrogen_dict.values())) for hydrogen_dict in hydrogen_per_bus_dict_all_years_vr.values()]
ax2.plot(horizon, [v for v in hydrogen_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU mean', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Hydrogen Prices [€/MWh]', fontsize=12)
ax2.set_title(variant, fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'Global Hydrogen Prices Over Time in EU (comparison)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
# if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    hydrogen_country = [hydrogen_dict[country] for hydrogen_dict in hydrogen_per_country_dict_all_years_dt.values()]
    ax1.plot(horizon, [v for v in hydrogen_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Difference in Hydrogen Prices [€/MWh]', fontsize=12)
ax1.set_title('Difference by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU mean ---
hydrogen_EU = [np.mean(list(hydrogen_dict.values())) for hydrogen_dict in hydrogen_per_bus_dict_all_years_dt.values()]
ax2.plot(horizon, [v for v in hydrogen_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU mean', linestyle='--')

ax2.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Difference in Hydrogen Prices [€/MWh]', fontsize=12)
ax2.set_title('EU Mean Difference', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Difference in Global Hydrogen Prices Over Time (= {variant} - {base})'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    hydrogen_country_dt = [hydrogen_dict[country] for hydrogen_dict in hydrogen_per_country_dict_all_years_dt.values()]
    hydrogen_country_bl = [hydrogen_dict[country] for hydrogen_dict in hydrogen_per_country_dict_all_years_bl.values()]
    ax1.plot(horizon, [v for v in np.array(hydrogen_country_dt)/np.array(hydrogen_country_bl)*100],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Difference in Hydrogen Prices [%]', fontsize=12)
ax1.set_title('Difference by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU mean ---
hydrogen_EU_dt = [np.mean(list(hydrogen_dict.values())) for hydrogen_dict in hydrogen_per_bus_dict_all_years_dt.values()]
hydrogen_EU_bl = [np.mean(list(hydrogen_dict.values())) for hydrogen_dict in hydrogen_per_bus_dict_all_years_bl.values()]
ax2.plot(horizon, [v for v in np.array(hydrogen_EU_dt)/np.array(hydrogen_EU_bl)*100],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU mean', linestyle='--')

ax2.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Difference in Hydrogen Prices [%]', fontsize=12)
ax2.set_title('EU Mean Difference', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Relative Difference in Global Hydrogen Prices Over Time (= {variant} - {base})'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
smr_produced_all_bl       = {c: [] for c in countries_of_interest}
smr_produced_all_bl['EU'] = []
lcoh_smr_bl       = {c: [] for c in countries_of_interest}
lcoh_smr_bl['EU'] = []

for year in horizon:
    df_smr_bl = get_hydrogen_smr(network_bl[year])
    smr_produced_bl = {country: 0  for country in countries_of_interest}
    lcoh_bl        = {country: [] for country in countries_of_interest}

    for i in df_smr_bl.index:
        if i[:2] in smr_produced_bl:
            smr_produced_bl[i[:2]] += df_smr_bl.loc[i, 'H2 Produced']*1e-6
            lcoh_bl[i[:2]].append(df_smr_bl.loc[i, 'LCOH'])

    for key in lcoh_bl:
        lcoh_bl[key] = np.mean(lcoh_bl[key])

    smr_produced_bl['EU'] = df_smr_bl['H2 Produced'].sum()*1e-6
    lcoh_bl['EU']        = np.mean(df_smr_bl['LCOH'])

    for c in countries_of_interest + ['EU']:
        smr_produced_all_bl[c].append(smr_produced_bl[c])
        lcoh_smr_bl[c].append(lcoh_bl[c])

In [ ]:
smr_produced_all_vr       = {c: [] for c in countries_of_interest}
smr_produced_all_vr['EU'] = []
smr_produced_all_dt       = {c: [] for c in countries_of_interest}
smr_produced_all_dt['EU'] = []
lcoh_smr_vr       = {c: [] for c in countries_of_interest}
lcoh_smr_vr['EU'] = []
lcoh_smr_dt       = {c: [] for c in countries_of_interest}
lcoh_smr_dt['EU'] = []

for year in horizon:
    df_smr_vr = get_hydrogen_smr(network_vr[year])
    smr_produced_vr = {country: 0  for country in countries_of_interest}
    lcoh_vr        = {country: [] for country in countries_of_interest}

    for i in df_smr_vr.index:
        if i[:2] in smr_produced_vr:
            smr_produced_vr[i[:2]] += df_smr_vr.loc[i, 'H2 Produced']*1e-6
            lcoh_vr[i[:2]].append(df_smr_vr.loc[i, 'LCOH'])

    for key in lcoh_vr:
        lcoh_vr[key] = np.mean(lcoh_vr[key])

    smr_produced_vr['EU'] = df_smr_vr['H2 Produced'].sum()*1e-6
    lcoh_vr['EU']        = np.mean(df_smr_vr['LCOH'])

    for c in countries_of_interest + ['EU']:
        smr_produced_all_vr[c].append(smr_produced_vr[c])
        lcoh_smr_vr[c].append(lcoh_vr[c])

smr_produced_all_dt, lcoh_smr_dt = {key: np.array(smr_produced_all_vr[key] - np.array(smr_produced_all_bl[key])) for key in smr_produced_all_bl}, {key: np.array(lcoh_smr_vr[key]) - np.array(lcoh_smr_bl[key]) for key in lcoh_smr_bl}	


In [ ]:
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    ax1.plot(horizon, smr_produced_all_bl[country],
             color=colors[i], lw=2, marker='o', markersize=6, 
             label=country)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('H₂ Produced [TWh]', fontsize=12)
ax1.set_title('H₂ Production by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
ax2.plot(horizon, smr_produced_all_bl['EU'], 
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('H₂ Produced [TWh]', fontsize=12)
ax2.set_title('EU Total H₂ Production', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Steam Methane Reforming Hydrogen Production ({base} scenario)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_bl}plots/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    ax1.plot(horizon, smr_produced_all_dt[country],
             color=colors[i], lw=2, marker='o', markersize=6, 
             label=country)

ax1.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Difference in H₂ Produced [TWh]', fontsize=12)
ax1.set_title('Difference by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
ax2.plot(horizon, smr_produced_all_dt['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Difference in H₂ Produced [TWh]', fontsize=12)
ax2.set_title(f'EU Total Difference (Cumulated = {sum(smr_produced_all_dt['EU']) / 1e0:.0f} TWh, {sum(smr_produced_all_dt["EU"])/sum(smr_produced_all_bl["EU"]) * 100:.1f}%)', fontsize=14, fontweight='bold')

ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Difference in Steam Methane Reforming Hydrogen Production Over Time (= {variant} - {base})'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    ax1.plot(horizon, np.array(smr_produced_all_dt[country])/np.array(smr_produced_all_bl[country]) * 100,
             color=colors[i], lw=2, marker='o', markersize=6, 
             label=country)

ax1.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Difference in H₂ Produced [%]', fontsize=12)
ax1.set_title('Difference by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.set_ylim([-150,150])

# --- Right plot: EU total ---
ax2.plot(horizon, np.array(smr_produced_all_dt['EU'])/np.array(smr_produced_all_bl['EU']) * 100,
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Difference in H₂ Produced [%]', fontsize=12)
ax2.set_title(f'EU Total Difference (Cumulated = {sum(smr_produced_all_dt["EU"])/sum(smr_produced_all_bl["EU"]) * 100:.1f}%)', fontsize=14, fontweight='bold')

ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.set_ylim([-30,30])

title = f'Relative Difference in Steam Methane Reforming Hydrogen Production Over Time (= {variant} - {base})'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
# if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

---
---
### $\text{RFNBO metrics}$ 
---
---

In [ ]:
h2_produced_all_bl       = {c: [] for c in countries_of_interest}
h2_produced_all_bl['EU'] = []
lcoh_all_bl              = {c: [] for c in countries_of_interest}
lcoh_all_bl['EU']        = []
lcoh_all_bl_capex        = {c: [] for c in countries_of_interest}
lcoh_all_bl_capex['EU']  = []
lcoh_all_bl_opex         = {c: [] for c in countries_of_interest}
lcoh_all_bl_opex['EU']   = []

for year in horizon:
    df_h2_bl = get_hydrogen_grid_connected(network_bl[year])
    h2_produced_bl = {country: 0  for country in countries_of_interest}
    lcoh_bl        = {country: [] for country in countries_of_interest}
    lcoh_bl_capex  = {country: [] for country in countries_of_interest}
    lcoh_bl_opex   = {country: [] for country in countries_of_interest}

    for i in df_h2_bl.index:
        if i[:2] in h2_produced_bl:
            h2_produced_bl[i[:2]] += df_h2_bl.loc[i, 'H2 Produced']*1e-6
            lcoh_bl[i[:2]].append(df_h2_bl.loc[i, 'LCOH'])
            lcoh_bl_capex[i[:2]].append(df_h2_bl.loc[i, 'LCOH_capex'])
            lcoh_bl_opex[i[:2]].append(df_h2_bl.loc[i, 'LCOH_opex'])

    for key in lcoh_bl:
        lcoh_bl[key] = np.mean(lcoh_bl[key])
        lcoh_bl_capex[key] = np.mean(lcoh_bl_capex[key])
        lcoh_bl_opex[key] = np.mean(lcoh_bl_opex[key])

    h2_produced_bl['EU'] = df_h2_bl['H2 Produced'].sum()*1e-6
    lcoh_bl['EU']        = np.mean(df_h2_bl['LCOH'])
    lcoh_bl_capex['EU']  = np.mean(df_h2_bl['LCOH_capex'])
    lcoh_bl_opex['EU']   = np.mean(df_h2_bl['LCOH_opex'])

    for c in countries_of_interest + ['EU']:
        h2_produced_all_bl[c].append(h2_produced_bl[c])
        lcoh_all_bl[c].append(lcoh_bl[c])
        lcoh_all_bl_capex[c].append(lcoh_bl_capex[c])
        lcoh_all_bl_opex[c].append(lcoh_bl_opex[c])

In [ ]:
h2_produced_all_vr       = {c: [] for c in countries_of_interest}
h2_produced_all_vr['EU'] = []
h2_produced_all_dt       = {c: [] for c in countries_of_interest}
h2_produced_all_dt['EU'] = []
lcoh_all_vr              = {c: [] for c in countries_of_interest}
lcoh_all_vr['EU']        = []
lcoh_all_vr_capex        = {c: [] for c in countries_of_interest}
lcoh_all_vr_capex['EU']  = []
lcoh_all_vr_opex         = {c: [] for c in countries_of_interest}
lcoh_all_vr_opex['EU']   = []
lcoh_all_dt              = {c: [] for c in countries_of_interest}
lcoh_all_dt['EU']        = []
lcoh_all_dt_capex        = {c: [] for c in countries_of_interest}
lcoh_all_dt_capex['EU']  = []
lcoh_all_dt_opex         = {c: [] for c in countries_of_interest}
lcoh_all_dt_opex['EU']   = []

for year in horizon:
    df_h2_vr = get_hydrogen_grid_connected(network_vr[year])
    h2_produced_vr = {country: 0  for country in countries_of_interest}
    lcoh_vr        = {country: [] for country in countries_of_interest}
    lcoh_vr_capex  = {country: [] for country in countries_of_interest}
    lcoh_vr_opex   = {country: [] for country in countries_of_interest}

    for i in df_h2_vr.index:
        if i[:2] in h2_produced_vr:
            h2_produced_vr[i[:2]] += df_h2_vr.loc[i, 'H2 Produced']*1e-6
            lcoh_vr[i[:2]].append(df_h2_vr.loc[i, 'LCOH'])
            lcoh_vr_capex[i[:2]].append(df_h2_vr.loc[i, 'LCOH_capex'])
            lcoh_vr_opex[i[:2]].append(df_h2_vr.loc[i, 'LCOH_opex'])

    for key in lcoh_vr:
        lcoh_vr[key] = np.mean(lcoh_vr[key])
        lcoh_vr_capex[key] = np.mean(lcoh_vr_capex[key])
        lcoh_vr_opex[key] = np.mean(lcoh_vr_opex[key])

    h2_produced_vr['EU'] = df_h2_vr['H2 Produced'].sum()*1e-6
    lcoh_vr['EU']        = np.mean(df_h2_vr['LCOH'])
    lcoh_vr_capex['EU']  = np.mean(df_h2_vr['LCOH_capex'])
    lcoh_vr_opex['EU']   = np.mean(df_h2_vr['LCOH_opex'])

    for c in countries_of_interest + ['EU']:
        h2_produced_all_vr[c].append(h2_produced_vr[c])
        lcoh_all_vr[c].append(lcoh_vr[c])
        lcoh_all_vr_capex[c].append(lcoh_vr_capex[c])
        lcoh_all_vr_opex[c].append(lcoh_vr_opex[c])

h2_produced_all_dt = {key: np.array(h2_produced_all_vr[key] - np.array(h2_produced_all_bl[key])) for key in h2_produced_all_bl}
lcoh_all_dt = {key: np.array(lcoh_all_vr[key]) - np.array(lcoh_all_bl[key]) for key in lcoh_all_bl}
lcoh_all_dt_capex = {key: np.array(lcoh_all_vr_capex[key]) - np.array(lcoh_all_bl_capex[key]) for key in lcoh_all_bl_capex}
lcoh_all_dt_opex = {key: np.array(lcoh_all_vr_opex[key]) - np.array(lcoh_all_bl_opex[key]) for key in lcoh_all_bl_opex}


In [ ]:
h2_produced_all_vr       = {c: [] for c in countries_of_interest}
h2_produced_all_vr['EU'] = []
h2_produced_all_dt       = {c: [] for c in countries_of_interest}
h2_produced_all_dt['EU'] = []
lcoh_all_vr       = {c: [] for c in countries_of_interest}
lcoh_all_vr['EU'] = []
lcoh_all_dt       = {c: [] for c in countries_of_interest}
lcoh_all_dt['EU'] = []

for year in horizon:
    df_h2_vr = get_hydrogen_grid_connected(network_vr[year])
    h2_produced_vr = {country: 0  for country in countries_of_interest}
    lcoh_vr        = {country: [] for country in countries_of_interest}

    for i in df_h2_vr.index:
        if i[:2] in h2_produced_vr:
            h2_produced_vr[i[:2]] += df_h2_vr.loc[i, 'H2 Produced']*1e-6
            lcoh_vr[i[:2]].append(df_h2_vr.loc[i, 'LCOH'])

    for key in lcoh_vr:
        lcoh_vr[key] = np.mean(lcoh_vr[key])

    h2_produced_vr['EU'] = df_h2_vr['H2 Produced'].sum()*1e-6
    lcoh_vr['EU']        = np.mean(df_h2_vr['LCOH'])

    for c in countries_of_interest + ['EU']:
        h2_produced_all_vr[c].append(h2_produced_vr[c])
        lcoh_all_vr[c].append(lcoh_vr[c])

h2_produced_all_dt, lcoh_all_dt = {key: np.array(h2_produced_all_vr[key] - np.array(h2_produced_all_bl[key])) for key in h2_produced_all_bl}, {key: np.array(lcoh_all_vr[key]) - np.array(lcoh_all_bl[key]) for key in lcoh_all_bl}

In [ ]:
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    ax1.plot(horizon, h2_produced_all_bl[country],
             color=colors[i], lw=2, marker='o', markersize=6, 
             label=country)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('H₂ Produced [TWh]', fontsize=12)
ax1.set_title('H₂ Production by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
ax2.plot(horizon, h2_produced_all_bl['EU'], 
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('H₂ Produced [TWh]', fontsize=12)
ax2.set_title('EU Total H₂ Production', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Grid Connected Hydrogen Production ({base} scenario)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_bl}plots/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    ax1.plot(horizon, h2_produced_all_vr[country],
             color=colors[i], lw=2, marker='o', markersize=6, 
             label=country)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('H₂ Produced [TWh]', fontsize=12)
ax1.set_title('H₂ Production by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
ax2.plot(horizon, h2_produced_all_vr['EU'], 
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('H₂ Produced [TWh]', fontsize=12)
ax2.set_title('EU Total H₂ Production', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Grid Connected Hydrogen Production ({variant} scenario)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_vr}plots/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: baseline ---
for i, country in enumerate(countries_of_interest):
    ax1.plot(horizon, h2_produced_all_bl[country],
             color=colors[i], lw=2, marker='o', markersize=6, 
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('H₂ Produced [TWh]', fontsize=12)
ax1.set_title(base, fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: variant---
for i, country in enumerate(countries_of_interest):
    ax2.plot(horizon, h2_produced_all_vr[country],
             color=colors[i], lw=2, marker='o', markersize=6, 
             label=country)

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('H₂ Produced [TWh]', fontsize=12)
ax2.set_title(variant, fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'Grid Connected Hydrogen Production Over Time by Country (comparison)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
# if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: baseline, EU ---
ax1.plot(horizon, h2_produced_all_bl['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('H₂ Produced [TWh]', fontsize=12)
ax1.set_title(base, fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: variant, EU ---
ax2.plot(horizon, h2_produced_all_vr['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('H₂ Produced [TWh]', fontsize=12)
ax2.set_title(variant, fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'Grid Connected Hydrogen Production Over Time in EU (comparison)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
# if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    ax1.plot(horizon, h2_produced_all_dt[country],
             color=colors[i], lw=2, marker='o', markersize=6, 
             label=country)

ax1.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Difference in H₂ Produced [TWh]', fontsize=12)
ax1.set_title('Difference by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
ax2.plot(horizon, h2_produced_all_dt['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Difference in H₂ Produced [TWh]', fontsize=12)
ax2.set_title(f'EU Total Difference (Cumulated = {sum(h2_produced_all_dt['EU']) / 1e0:.0f} TWh, {sum(h2_produced_all_dt["EU"])/sum(h2_produced_all_bl["EU"]) * 100:.1f}%)', fontsize=14, fontweight='bold')

ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Difference in Grid Connected Hydrogen Production Over Time (= {variant} - {base})'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    ax1.plot(horizon, np.array(h2_produced_all_dt[country])/np.array(h2_produced_all_bl[country]) * 100,
             color=colors[i], lw=2, marker='o', markersize=6, 
             label=country)

ax1.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Difference in H₂ Produced [%]', fontsize=12)
ax1.set_title('Difference by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.set_ylim([-150,150])

# --- Right plot: EU total ---
ax2.plot(horizon, np.array(h2_produced_all_dt['EU'])/np.array(h2_produced_all_bl['EU']) * 100,
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Difference in H₂ Produced [%]', fontsize=12)
ax2.set_title(f'EU Total Difference (Cumulated = {sum(h2_produced_all_dt["EU"])/sum(h2_produced_all_bl["EU"]) * 100:.1f}%)', fontsize=14, fontweight='bold')

ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.set_ylim([-40,40])

title = f'Relative Difference in Grid Connected Hydrogen Production Over Time (= {variant} - {base})'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
# if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    ax1.plot(horizon, lcoh_all_bl[country],
             color=colors[i], lw=2, marker='o', markersize=6, 
             label=country)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('LCOH [€/MWh]', fontsize=12)
ax1.set_title('LCOH by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
#ax1.set_ylim([90,120])

# --- Right plot: EU total ---
ax2.plot(horizon, lcoh_all_bl['EU'], 
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU mean', linestyle='--')
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('LCOH [€/MWh]', fontsize=12)
ax2.set_title('EU Mean LCOH', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Levelized Cost of Grid Connected Hydrogen ({base} scenario)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_bl}plots/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    ax1.plot(horizon, lcoh_all_bl[country],
             color=colors[i], lw=2, marker='o', markersize=6, 
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('LCOH [€/MWh]', fontsize=12)
ax1.set_title(base, fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
for i, country in enumerate(countries_of_interest):
    ax2.plot(horizon, lcoh_all_vr[country],
             color=colors[i], lw=2, marker='o', markersize=6, 
             label=country)

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('LCOH [€/MWh]', fontsize=12)
ax2.set_title(variant, fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'Levelized Cost of Grid Connected Hydrogen Over Time by Country (comparison)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
# if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: baseline, EU ---
ax1.plot(horizon, lcoh_all_bl['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU mean', linestyle='--')

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('LCOH [€/MWh]', fontsize=12)
ax1.set_title(base, fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: variant, EU ---
ax2.plot(horizon, lcoh_all_vr['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU mean', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('LCOH [€/MWh]', fontsize=12)
ax2.set_title(variant, fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = 'Levelized Cost of Grid Connected Hydrogen Over Time in EU (comparison)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
# if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    ax1.plot(horizon, lcoh_all_dt[country],
             color=colors[i], lw=2, marker='o', markersize=6, 
             label=country)

ax1.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Difference in LCOH [€/MWh]', fontsize=12)
ax1.set_title('Difference by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
#ax1.set_ylim([-20,20])

# --- Right plot: EU total ---
ax2.plot(horizon, lcoh_all_dt['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU mean', linestyle='--')

ax2.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Difference in LCOH [€/MWh]', fontsize=12)
ax2.set_title('EU Mean Difference', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
#ax2.set_ylim([-5,5])

title = f'Difference in Levelized Cost of Grid Connected Hydrogen Over Time (= {variant} - {base})'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    ax1.plot(horizon, np.array(lcoh_all_dt[country])/np.array(lcoh_all_bl[country]) * 100,
             color=colors[i], lw=2, marker='o', markersize=6, 
             label=country)

ax1.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Difference in LCOH [%]', fontsize=12)
ax1.set_title('Difference by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU mean ---
ax2.plot(horizon, np.array(lcoh_all_dt['EU'])/np.array(lcoh_all_bl['EU']) * 100,
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU mean', linestyle='--')

ax2.axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Difference in LCOH [%]', fontsize=12)
ax2.set_title('EU Mean Difference', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Relative Difference in Levelized Cost of Grid Connected Hydrogen Over Time (= {variant} - {base})'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
for my_country in countries_of_interest:

       fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

       color_index = countries_of_interest.index(my_country)
       width = 0.35
       hatch_capex = ".."      # solid
       hatch_opex  = "//"    # hatched
       x = np.arange(len(horizon))

       # --- Left plot: breakdown ---

       # Baseline (left bars, transparent)
       ax1.bar(x - width/2, lcoh_all_bl_capex[my_country], width,
              color=colors[color_index], alpha=0.3, hatch=hatch_capex,
              edgecolor="white", label=f"{base} CAPEX")
       ax1.bar(x - width/2, lcoh_all_bl_opex[my_country], width,
              bottom=lcoh_all_bl_capex[my_country],
              color=colors[color_index], alpha=0.3, hatch=hatch_opex,
              edgecolor="white", label=f"{base} OPEX")

       # Variant (right bars, full opacity)
       ax1.bar(x + width/2, lcoh_all_vr_capex[my_country], width,
              color=colors[color_index], alpha=1.0, hatch=hatch_capex,
              edgecolor="black", label=f"{variant} CAPEX")
       ax1.bar(x + width/2, lcoh_all_vr_opex[my_country], width,
              bottom=lcoh_all_vr_capex[my_country],
              color=colors[color_index], alpha=1.0, hatch=hatch_opex,
              edgecolor="black", label=f"{variant} OPEX")

       ax1.set_xlabel('Year', fontsize=12)
       ax1.set_ylabel('LCOH [€/MWh]', fontsize=12)
       ax1.set_title('LCOH breakdown', fontsize=14, fontweight='bold')
       ax1.legend(fontsize=11, framealpha=0.9)
       ax1.grid(alpha=0.3, linestyle='--')
       ax1.set_xticks(x)
       ax1.set_xticklabels(horizon)
       ax1.spines['top'].set_visible(False)
       ax1.spines['right'].set_visible(False)

       # --- Right plot: delta ---

       ax2.bar(x - width/2, lcoh_all_dt_capex[my_country], width,
              color="grey", alpha=1.0, hatch=hatch_capex,
              edgecolor="black", label="CAPEX")
       ax2.bar(x + width/2, lcoh_all_dt_opex[my_country], width,
              color="grey", alpha=1.0, hatch=hatch_opex,
              edgecolor="black", label="OPEX")

       ax2.set_xlabel('Year', fontsize=12)
       ax2.set_ylabel('LCOH [€/MWh]', fontsize=12)
       ax2.set_title('LCOH breakdown difference', fontsize=14, fontweight='bold')
       ax2.legend(fontsize=11, framealpha=0.9)
       ax2.grid(alpha=0.3, linestyle='--')
       ax2.set_xticks(x)
       ax2.set_xticklabels(horizon)
       ax2.spines['top'].set_visible(False)
       ax2.spines['right'].set_visible(False)

       title = f'{my_country} Breakdown of Levelized Cost of Grid Connected Hydrogen Over Time (= {variant} - {base})'
       plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
       plt.tight_layout()
       if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
       plt.show()

---
---
### $\text{Exemption criteria}$
---
---

In [ ]:
renewable_share_per_country_dict_all_years_bl_prod = {year : {} for year in horizon}
grid_co2_per_country_dict_all_years_bl_prod = {year : {} for year in horizon}

for year in horizon:
    if base == 'baseline':
        out = get_vre_share_carbon_intensity_prod(network_bl[year], config_bl, base, year)
    else:
        out = get_vre_share_carbon_intensity_prod(network_bl[year], config_bl, f'RFNBO_{base}', year)

    renewable_share_per_country_dict_all_years_bl_prod[year] = {
        country: values['renewable_share']
        for country, values in out.items()
    }
    grid_co2_per_country_dict_all_years_bl_prod[year]= {
        country: values['co2_intensity']
        for country, values in out.items()
    }

In [ ]:
renewable_share_per_country_dict_all_years_vr_prod = {year : {} for year in horizon}
grid_co2_per_country_dict_all_years_vr_prod = {year : {} for year in horizon}

for year in horizon:
    if base == 'baseline':
        out = get_vre_share_carbon_intensity_prod(network_vr[year], config_vr, base, year)
    else:
        out = get_vre_share_carbon_intensity_prod(network_vr[year], config_vr, f'RFNBO_{base}', year)

    renewable_share_per_country_dict_all_years_vr_prod[year] = {
        country: values['renewable_share']
        for country, values in out.items()
    }
    grid_co2_per_country_dict_all_years_vr_prod[year]= {
        country: values['co2_intensity']
        for country, values in out.items()
    }

renewable_share_per_country_dict_all_years_dt_prod = {
    year: {
        country: renewable_share_per_country_dict_all_years_vr_prod[year][country]
                  - renewable_share_per_country_dict_all_years_bl_prod[year][country]
        for country in renewable_share_per_country_dict_all_years_vr_prod[year]
    }
    for year in renewable_share_per_country_dict_all_years_vr_prod
}
grid_co2_per_country_dict_all_years_dt_prod = {
    year: {
        country: grid_co2_per_country_dict_all_years_vr_prod[year][country]
                  - grid_co2_per_country_dict_all_years_bl_prod[year][country]
        for country in grid_co2_per_country_dict_all_years_vr_prod[year]
    }
    for year in grid_co2_per_country_dict_all_years_vr_prod
}

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: production ---
for i, country in enumerate(countries_of_interest):
    grid_co2_country = [grid_co2_dict[country] for grid_co2_dict in grid_co2_per_country_dict_all_years_dt_prod.values()]
    ax1.plot(horizon, [v for v in grid_co2_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Difference in Grid CO2 Intensity [gCO2/MJ]', fontsize=12)
ax1.set_title('Difference in Grid CO2 Intensity (Production)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: consumption ---
for i, country in enumerate(countries_of_interest):
    renewable_share_country = [renewable_share_dict[country] for renewable_share_dict in renewable_share_per_country_dict_all_years_dt_prod.values()]
    ax2.plot(horizon, [v for v in renewable_share_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country, linestyle='-.')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Difference in Renewable Share [%]', fontsize=12)
ax2.set_title('Difference in Renewable Share (Production)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Difference in Exemption Criteria Over Time (= {variant} - {base})'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
title = f'Difference in Exemption Criteria Over Time'
if save_fig: plt.savefig(f'{path_dt}/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
renewable_share_per_country_dict_all_years_bl_cons = {year : {} for year in horizon}
grid_co2_per_country_dict_all_years_bl_cons = {year : {} for year in horizon}

for year in horizon:
    if base == 'baseline':
        out = get_vre_share_carbon_intensity_cons(network_bl[year], config_bl, base, year)
    else:
        out = get_vre_share_carbon_intensity_cons(network_bl[year], config_bl, f'RFNBO_{base}', year)
    
    renewable_share_per_country_dict_all_years_bl_cons[year] = {
        country: values['renewable_share']
        for country, values in out.items()
    }
    grid_co2_per_country_dict_all_years_bl_cons[year]= {
        country: values['co2_intensity']
        for country, values in out.items()
    }

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: production ---
for i, country in enumerate(countries_of_interest):
    grid_co2_country = [grid_co2_dict[country] for grid_co2_dict in grid_co2_per_country_dict_all_years_bl_prod.values()]
    ax1.plot(horizon, [v for v in grid_co2_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.axhline(18, color='darkgrey', linewidth=1.5, zorder=0)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Grid CO2 Intensity [gCO2/MJ]', fontsize=12)
ax1.set_title('Grid CO2 Intensity by Country (Production)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.set_ylim([-3,100])

# --- Right plot: consumption ---
for i, country in enumerate(countries_of_interest):
    grid_co2_country = [grid_co2_dict[country] for grid_co2_dict in grid_co2_per_country_dict_all_years_bl_cons.values()]
    ax2.plot(horizon, [v for v in grid_co2_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country, linestyle='-.')

ax2.axhline(18, color='darkgrey', linewidth=1.5, zorder=0)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Grid CO2 Intensity [gCO2/MJ]', fontsize=12)
ax2.set_title('Grid CO2 Intensity by Country (Consumption)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Grid CO2 Intensity Over Time ({base} scenario)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_bl}plots/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: production ---
for i, country in enumerate(countries_of_interest):
    renewable_share_country = [renewable_share_dict[country] for renewable_share_dict in renewable_share_per_country_dict_all_years_bl_prod.values()]
    ax1.plot(horizon, [v for v in renewable_share_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.axhline(90, color='darkgrey', linewidth=1.5, zorder=0)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Renewable Share [%]', fontsize=12)
ax1.set_title('Renewable Share by Country (Production)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.set_ylim([49,101])

# --- Right plot: consumption ---
for i, country in enumerate(countries_of_interest):
    renewable_share_country = [renewable_share_dict[country] for renewable_share_dict in renewable_share_per_country_dict_all_years_bl_cons.values()]
    ax2.plot(horizon, [v for v in renewable_share_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country, linestyle='--')

ax2.axhline(90, color='darkgrey', linewidth=1.5, zorder=0)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Renewable Share [%]', fontsize=12)
ax2.set_title('Renewable Share by Country (Consumption)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Renewable Share Over Time ({base} scenario)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_bl}plots/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
em_2025_share = {
    'AT': 73, 
    'BE': 39, 
    'BG': 28, 
    'CH': 63, 
    'CZ': 18, 
    'DE': 60, 
    'DK': (80+86)/2, 
    'EE': 59, 
    'ES': 56, 
    'FI': 59, 
    'FR': 28, 
    'GB': 53, 
    'GR': 47, 
    'HR': 58, 
    'HU': 27, 
    'IE': 47, 
    'IT': (38+54+49+49+53)/5, 
    'LT': 73, 
    'LU': 57, 
    'LV': 70, 
    'NL': 54, 
    'NO': 95, 
    'PL': 31, 
    'PT': 76, 
    'RO': 41, 
    'SE': (72+64)/2, 
    'SI': 48, 
    'SK': 18,
}

em_2025_intensity = {
    'AT': 164, 
    'BE': 173, 
    'BG': 366, 
    'CH': 50, 
    'CZ': 439, 
    'DE': 342, 
    'DK': (99+128)/2, 
    'EE': 243, 
    'ES': 135, 
    'FI': 68, 
    'FR': 32, 
    'GB': 174, 
    'GR': 297, 
    'HR': 221, 
    'HU': 232, 
    'IE': 301, 
    'IT': (274+250+295+294+281)/5, 
    'LT': 135, 
    'LU': 211, 
    'LV': 155, 
    'NL': 255, 
    'NO': 29, 
    'PL': 566, 
    'PT': 118, 
    'RO': 300, 
    'SE': (36+20)/2, 
    'SI': 194, 
    'SK': 232,
}

In [ ]:
# --- Extract data for countries of interest ---
labels = countries_of_interest
real_data  = [em_2025_share.get(c) for c in labels]
prod_data  = [renewable_share_per_country_dict_all_years_bl_prod[2025].get(c) for c in labels]
cons_data  = [renewable_share_per_country_dict_all_years_bl_cons[2025].get(c) for c in labels]

# --- Plot ---
x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(16, 6))

bars1 = ax.bar(x - width, real_data,  width, label='Real life (2025)',              color='#2a78d6')
bars2 = ax.bar(x,          prod_data,  width, label='Model — production-based',      color='#1baf7a')
bars3 = ax.bar(x + width,  cons_data,  width, label='Model — consumption-based',     color='#eda100')

ax.set_ylabel('Renewable electricity share (%)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 100)
ax.legend()
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
ax.set_ylim(7,87)

plt.tight_layout()
#if save_fig: plt.savefig('renewable_share_comparison.png', dpi=150)
plt.show()

In [ ]:
# --- Extract data for countries of interest ---
labels = countries_of_interest
real_data  = [em_2025_intensity.get(c) for c in labels]
prod_data  = [grid_co2_per_country_dict_all_years_bl_prod[2025].get(c)*3.6 for c in labels]
cons_data  = [grid_co2_per_country_dict_all_years_bl_cons[2025].get(c)*3.6 for c in labels]

# --- Plot ---
x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(16, 6))

bars1 = ax.bar(x - width, real_data,  width, label='Real life (2025)',              color='#2a78d6')
bars2 = ax.bar(x,          prod_data,  width, label='Model — production-based',      color='#1baf7a')
bars3 = ax.bar(x + width,  cons_data,  width, label='Model — consumption-based',     color='#eda100')

ax.set_ylabel('Grid intensity (gCO2/kWh)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
#if save_fig: plt.savefig('renewable_share_comparison.png', dpi=150)
plt.show()

---
---
### $\text{Violation of temporal correlation}$
---
---

In [ ]:
# delta_VRE - electrolysers : check number of negative hours, count share of production form non renewable

---
---
### $\text{Old out}$
---
---

In [ ]:
def get_renewable_share(n, config):
    electricity_buses_names = n.buses[(n.buses.carrier == 'AC')].index

    #Get net export/import per bus
    net_exchange = {bus:0 for bus in electricity_buses_names}

    for line in n.lines.index:
        b0 = n.lines.at[line, "bus0"]
        b1 = n.lines.at[line, "bus1"]

        if b0 in electricity_buses_names:
            net_exchange[b0] += (n.snapshot_weightings.generators @ -n.lines_t.p0[line])
        if b1 in electricity_buses_names:
            net_exchange[b1] += (n.snapshot_weightings.generators @ -n.lines_t.p1[line])
        
    #Get gross eletricity production per bus
    generator_types = list(
      set(config["electricity"]["renewable_carriers"] + ["solar rooftop","ror"]))


    conv_types = list(
        set(config["electricity"]["conventional_carriers"] + ["urban central gas CHP",
                "urban central gas CHP CC","urban central solid biomass CHP","urban central solid biomass CHP CC",
                "H2 Fuel Cell","H2 turbine","geothermal organic rankine cycle"]))


    gens = n.generators.index[
        n.generators.carrier.isin(generator_types)
        ]

    links = n.links.index[
        n.links.carrier.isin(conv_types)
        ]
        
    hydro= n.storage_units.index[
    n.storage_units.carrier == "hydro"
    ]

    gen = (n.snapshot_weightings.generators @ n.generators_t.p[gens])


    link = (n.snapshot_weightings.generators @ -n.links_t.p1[links])


    hyd = (n.snapshot_weightings.generators @ n.storage_units_t.p_dispatch[hydro])

    total_gen = pd.concat([gen, link, hyd])

    gross_electricity_prod_per_bus = {bus:0 for bus in electricity_buses_names}

    for index, row in total_gen.items():
        gross_electricity_prod_per_bus[index[:5]] += row

    
    #Get net electricity production per bus
    net_electricity_prod_per_bus = {bus:0 for bus in electricity_buses_names}
    for key, item in net_electricity_prod_per_bus.items():
        net_electricity_prod_per_bus[key] = gross_electricity_prod_per_bus[key] + net_exchange[key]

    
    #Get renewable production per bus (biomass included!)
    renewable_carriers = generator_types + [
    "urban central solid biomass CHP",
    "urban central solid biomass CHP CC",
    "geothermal organic rankine cycle"]

    pattern = "|".join(renewable_carriers)
    renewable_total = total_gen[total_gen.index.to_series().str.contains(pattern, regex=True)]

    renewable_prod_per_bus = {bus:0 for bus in electricity_buses_names}
    for index, row in renewable_total.items():
        renewable_prod_per_bus[index[:5]] += row

    
    #Get renewable share per bus
    renewable_share_per_bus = {bus:0 for bus in electricity_buses_names}
    for key, item in renewable_share_per_bus.items():
        renewable_share_per_bus[key] = renewable_prod_per_bus[key]/net_electricity_prod_per_bus[key]*100

    
    #Get renewable share per country
    countries = n.buses.country.unique()
    countries = countries[(countries != "EU") & (countries != "")]

    renewable_prod_per_country = {country:0 for country in countries}
    net_electricity_prod_per_country = {country:0 for country in countries}
    renewable_share_per_country = {country:0 for country in countries}


    for key, item in renewable_prod_per_bus.items():
        renewable_prod_per_country[key[:2]] += item

    for key, item in net_electricity_prod_per_bus.items():
        net_electricity_prod_per_country[key[:2]] += item

    for key, item in renewable_share_per_country.items():
        renewable_share_per_country[key] = renewable_prod_per_country[key]/net_electricity_prod_per_country[key]*100
    renewable_share_per_country['EU'] = sum(renewable_prod_per_country.values())/sum(net_electricity_prod_per_country.values())*100

    
    return renewable_share_per_country


In [ ]:
def get_grid_co2(n, config):

    electricity_buses_names = n.buses[(n.buses.carrier == 'AC')].index

    #Get gross eletricity production per bus
    generator_types = list(
        set(config["electricity"]["renewable_carriers"] + ["solar rooftop","ror"]))


    conv_types = list(
        set(config["electricity"]["conventional_carriers"] + ["urban central gas CHP",
                "urban central gas CHP CC","urban central solid biomass CHP","urban central solid biomass CHP CC",
                "H2 Fuel Cell","H2 turbine","geothermal organic rankine cycle"]))


    gens = n.generators.index[
        n.generators.carrier.isin(generator_types)
        ]

    links = n.links.index[
        n.links.carrier.isin(conv_types)
        ]
        
    hydro= n.storage_units.index[
    n.storage_units.carrier == "hydro"
    ]

    gen = (n.snapshot_weightings.generators @ n.generators_t.p[gens])


    link = (n.snapshot_weightings.generators @ -n.links_t.p1[links])


    hyd = (n.snapshot_weightings.generators @ n.storage_units_t.p_dispatch[hydro])

    total_gen = pd.concat([gen, link, hyd])

    gross_electricity_prod_per_bus = {bus:0 for bus in electricity_buses_names}

    for index, row in total_gen.items():
        gross_electricity_prod_per_bus[index[:5]] += row

    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    countries_list = countries_list[countries_list != 'EU']
    gross_electricity_prod_per_country = {country:0 for country in countries_list}

    for key, item in gross_electricity_prod_per_bus.items():
        gross_electricity_prod_per_country[key[:2]] += item

    # Get the co2 emissions, related to production
    co2_links_bus1 = n.links[
        n.links.bus1.str.contains('co2 atmosphere', case=False, na=False) 
    ]
    co2_links_bus2 = n.links[
        n.links.bus2.str.contains('co2 atmosphere', case=False, na=False) 
    ]
    co2_links_bus3 = n.links[
        n.links.bus3.str.contains('co2 atmosphere', case=False, na=False) 
    ]

    # CO2 from bus1 links
    if not co2_links_bus1.empty:
        co2_bus1 = (
            -n.links_t.p1[co2_links_bus1.index]
            .multiply(n.snapshot_weightings.generators, axis=0)
        )
    # CO2 from bus2 links
    if not co2_links_bus2.empty:
        co2_bus2 = (
            -n.links_t.p2[co2_links_bus2.index]
            .multiply(n.snapshot_weightings.generators, axis=0)
        )
    # CO2 from bus3 links
    if not co2_links_bus3.empty:
        co2_bus3 = (
            -n.links_t.p3[co2_links_bus3.index]
            .multiply(n.snapshot_weightings.generators, axis=0)
        )

    co2_all_buses = pd.concat((co2_bus1, co2_bus2, co2_bus3), axis=1).sum(axis = 0)

    co2_links = co2_all_buses.loc[co2_all_buses.index.intersection(links)]

    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    grid_co2_country = {country: 0 for country in countries_list}

    electricity_buses_names = n.buses[(n.buses.carrier == 'AC')].index
    grid_co2_bus = {bus: 0 for bus in electricity_buses_names}


    for i in co2_links.index:
        grid_co2_bus[i[:5]] += co2_links.loc[i]

    for i in co2_links.index:
        grid_co2_country[i[:2]] += co2_links.loc[i]
        grid_co2_country['EU'] += co2_links.loc[i]

    for k in grid_co2_bus:
        grid_co2_bus[k] /= gross_electricity_prod_per_bus[k]
        grid_co2_bus[k] *= 1000 #Converting from t/MWh to g/kWh
        grid_co2_bus[k] /= 3.6 #Converting from g/kWh to g/MJ (1 kWh is 3.6 MJ)

    for k in grid_co2_country:
        if k != 'EU':
            grid_co2_country[k] /= gross_electricity_prod_per_country[k]
            grid_co2_country[k] *= 1000 #Converting from t/MWh to g/kWh
            grid_co2_country[k] /= 3.6 #Converting from g/kWh to g/MJ (1 kWh is 3.6 MJ)

    grid_co2_country['EU'] /= (sum(gross_electricity_prod_per_country.values()) * 1000 / 3.6)

    return grid_co2_country

In [ ]:
renewable_share_per_country_dict_all_years_bl = {year : {} for year in horizon}

for year in horizon:
    renewable_share_per_country_dict_all_years_bl[year]= get_renewable_share(network_bl[year], config_bl)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    renewable_share_country = [renewable_share_dict[country] for renewable_share_dict in renewable_share_per_country_dict_all_years_bl.values()]
    ax1.plot(horizon, [v for v in renewable_share_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Renewable Share [%]', fontsize=12)
ax1.set_title('Renewable Share by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
for i, country in enumerate(['EU']):
    renewable_share_country = [renewable_share_dict[country] for renewable_share_dict in renewable_share_per_country_dict_all_years_bl.values()]
    ax2.plot(horizon, [v for v in renewable_share_country],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Renewable Share [%]', fontsize=12)
ax2.set_title('EU Renewable Share', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Grid Renewable Share Over Time ({base} scenario)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
# if save_fig: plt.savefig(f'{path_bl}plots/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
grid_co2_per_country_dict_all_years_bl = {year : {} for year in horizon}

for year in horizon:
    grid_co2_per_country_dict_all_years_bl[year]= get_grid_co2(network_bl[year], config_bl)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Left plot: countries ---
for i, country in enumerate(countries_of_interest):
    grid_co2_country = [grid_co2_dict[country] for grid_co2_dict in grid_co2_per_country_dict_all_years_bl.values()]
    ax1.plot(horizon, [v for v in grid_co2_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Grid CO2 Intensity [gCO2/MJ]', fontsize=12)
ax1.set_title('Grid CO2 Intensity by Country', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.3, linestyle='--')
ax1.set_xticks(horizon)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Right plot: EU total ---
for i, country in enumerate(['EU']):
    grid_co2_country = [grid_co2_dict[country] for grid_co2_dict in grid_co2_per_country_dict_all_years_bl.values()]
    ax2.plot(horizon, [v for v in grid_co2_country],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Grid CO2 Intensity [gCO2/MJ]', fontsize=12)
ax2.set_title('EU Grid CO2 Intensity', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xticks(horizon)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

title = f'Grid CO2 Intensity Over Time ({base} scenario)'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
# if save_fig: plt.savefig(f'{path_bl}plots/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()